# Preprocesamiento

Ante los hallazgos encontrados, consultamos con el especialista en el tema y principal stakeholder del proyecto.
Validamos las recomendaciones sobre los límites temporales a considerar, consultamos por los motivos que podían provocar las filas vacias, y consultamos por el corte alrededor de los 15000 m.

Respecto a perfiles con valores 0, efectivamente los archivos originales eran el crudo, nos indicó que los dascartáramos.
Respecto a los ceros de perfiles con valores, nos indicó que tomáramos unicamente valores entre los 15000 y 80000 metros. Nos comentó que el sistema filtra observaciones dudosas, que en general, a bajas alturas, las partículas en suspensión en la atmósfera (sólidas o líquidas) contaminan los perfiles.
Por otro lado nos recordó que la intención es procesar los datos del 2022, 2023 y 2024.

En función de los comentarios del especialista vamos a proceder a lo siguiente:

1. Eliminar perfiles completamente vacíos
2. Filtrar período 2022-2024
3. Conservar únicamente alturas entre 15000 y 80000 m
4. Conversión de ceros a NaN
5. Guardar dataset resultante

In [1]:
# Vamos a volver a cargar el dataset

import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(r"C:\Users\orlan\OneDrive\Documents\Tecnicatura\2A1C - Aprendizaje Automatico\Estacion Astronomica RG - Lidar GW\data\temperaturas_unificadas.csv", 
                 encoding= "ISO-8859-1")


In [2]:
print("Dimensiones del dataset original: ",df.shape)

Dimensiones del dataset original:  (40747, 1603)


In [3]:
# Eliminar perfiles completamente vacíos

temp_cols = df.columns[3:]

mask_ceros = (
    df[temp_cols]
    .eq(0)
    .all(axis=1)
)

df = df[~mask_ceros].copy()

print("Perfiles descartados (ceros):", mask_ceros.sum())

print("Perfiles restantes sin ceros:",df.shape)

Perfiles descartados (ceros): 15055
Perfiles restantes sin ceros: (25692, 1603)


In [4]:
# Filtrar período 2022-2024

# asegurar formato fecha
df["fecha"] = pd.to_datetime(df["fecha"])

# conservar únicamente 2022-2024
df = df[
    (df["fecha"].dt.year >= 2022)
    & (df["fecha"].dt.year <= 2024)
].copy()

print("Perfiles restantes sin ceros 2022-2024:",df.shape)

Perfiles restantes sin ceros 2022-2024: (11041, 1603)


In [6]:
# Conservar únicamente alturas entre 15000 y 80000 m

alturas = pd.Index(temp_cols).astype(float)

alturas_validas = [
    col
    for col in temp_cols
    if 15000 <= float(col) <= 80000]

df_limpio = df[
    ["archivo", "fecha", "time"]
    + alturas_validas
].copy()

print("Perfiles restantes sin ceros 2022-2024 15 a 80 k:",df_limpio.shape)

print(
    "Altura mínima:",min(map(float, alturas_validas)),
    "Altura máxima:",max(map(float, alturas_validas)))


Perfiles restantes sin ceros 2022-2024 15 a 80 k: (11041, 653)
Altura mínima: 15068.0 Altura máxima: 79968.0


In [8]:
# Conversión de ceros a NaN

import numpy as np

temp_cols = df_limpio.columns[3:]

df_limpio[temp_cols] = (
    df_limpio[temp_cols]
    .replace(0, np.nan))

print(
    df_limpio[temp_cols]
    .eq(0)
    .sum()
    .sum()
)

0


In [9]:
print("Dataset final:", df_limpio.shape)

Dataset final: (11041, 653)


In [10]:
df_limpio.to_csv(
    "perfiles_limpios_2022_2024_15000_80000.csv",
    index=False
)

print("Archivo guardado.")

Archivo guardado.
